In [ ]:
# CISB5123 Text Analytics
# Lab Assignment 3 - Topic Modeling
# Sarah binti Shamsul Kamar (BIS01084600)

# 1. Import necessary libraries
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer

# For data handling and topic modeling
import pandas as pd
import re
from gensim import corpora
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel

# Download NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')


# 2. Read the data (use only the 'text' column)

# Load the dataset
df = pd.read_csv('news_dataset.csv')

# Extract only the 'text' column
documents = df['text'].tolist()

# 3. Perform text pre-processing
# Remove null values
initial_count = len(documents)
documents = [doc for doc in documents if pd.notna(doc)]
print(f"\nRemoved {initial_count - len(documents)} null values")

# Initialize preprocessing tools
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """
    Preprocess a single text document:
    - Convert to lowercase
    - Remove punctuation and numbers
    - Tokenize
    - Remove stopwords
    - Apply stemming and lemmatization
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    tokens = [token for token in tokens if token not in stop_words]
    
    # Remove short words (length < 3)
    tokens = [token for token in tokens if len(token) > 2]
    
    # Apply stemming and lemmatization
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in stemmed_tokens]
    
    # Remove duplicates while preserving order
    seen = set()
    unique_tokens = []
    for token in lemmatized_tokens:
        if token not in seen:
            seen.add(token)
            unique_tokens.append(token)
    
    return unique_tokens

# Preprocess all documents
print("\nPreprocessing documents...")
preprocessed_documents = [preprocess_text(doc) for doc in documents]

# Filter out empty documents
non_empty_docs = [(i, doc) for i, doc in enumerate(preprocessed_documents) if len(doc) > 0]
preprocessed_documents = [doc for doc in preprocessed_documents if len(doc) > 0]
documents = [documents[i] for i, _ in non_empty_docs]

print(f"Number of non-empty documents after preprocessing: {len(preprocessed_documents)}")


# 4. Perform LDA using Gensim
# Create a Gensim Dictionary object from the preprocessed documents
dictionary = corpora.Dictionary(preprocessed_documents)

# Filter out extreme tokens:
# - no_below: tokens that appear in less than 5 documents
# - no_above: tokens that appear in more than 60% of documents
dictionary.filter_extremes(no_below=5, no_above=0.6)

print(f"\nDictionary size after filtering: {len(dictionary)}")

# Convert each preprocessed document into a bag-of-words representation
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

# Train LDA model with 4 topics
lda_model = LdaModel(
    corpus=corpus,
    num_topics=4,
    id2word=dictionary,
    passes=20,
    random_state=42,
    alpha='auto',
    eta='auto'
)

print("\nLDA Model training completed!")

# 5. Evaluate the LDA model using Coherence score
# Compute coherence score
coherence_model_lda = CoherenceModel(
    model=lda_model,
    texts=preprocessed_documents,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model_lda.get_coherence()
print(f"\nCoherence Score: {coherence_score:.4f}")

# 6. Interpret the results
print("TOPIC MODELING RESULTS")

# Print the top terms for each topic
print("\nTop terms for each topic:")

for idx in range(4):
    print(f"\nTopic #{idx}:")
    top_terms = lda_model.show_topic(idx, top=10)
    terms_with_weights = []
    for term, weight in top_terms:
        print(f"  - {term} (weight: {weight:.4f})")
        terms_with_weights.append(f"{term} ({weight:.3f})")
    
# Assign dominant topic to each document
document_topics = []
for i, doc_bow in enumerate(corpus):
    topics = lda_model.get_document_topics(doc_bow)
    if topics:
        dominant_topic = max(topics, key=lambda x: x[1])[0]
        confidence = max(topics, key=lambda x: x[1])[1]
    else:
        dominant_topic = -1
        confidence = 0
    document_topics.append({
        'document_index': i,
        'dominant_topic': dominant_topic,
        'confidence': confidence
    })

# Create DataFrame with results
results_df = pd.DataFrame({
    'Original_Text': documents,
    'Preprocessed_Tokens': preprocessed_documents,
    'Dominant_Topic': [dt['dominant_topic'] for dt in document_topics],
    'Confidence': [dt['confidence'] for dt in document_topics]
})

# Topic distribution summary
print("TOPIC DISTRIBUTION SUMMARY")
topic_counts = results_df['Dominant_Topic'].value_counts().sort_index()
for topic, count in topic_counts.items():
    percentage = (count / len(results_df)) * 100
    print(f"Topic #{topic}: {count} documents ({percentage:.1f}%)")


# INTERPRETATION OF COHERENCE SCORE
print("INTERPRETATION OF COHERENCE SCORE")
print(f"""
The coherence score achieved by this LDA model is {coherence_score:.4f}.

Coherence score interpretation:
- Scores range from 0 to 1, where higher scores indicate more semantically 
  coherent and interpretable topics.
- A score between 0.4 - 0.5 is considered good for topic models.
- A score between 0.5 - 0.6 indicates very good coherence.
- Scores above 0.6 are excellent.

With a coherence score of {coherence_score:.4f}, this model suggests that:
- The {coherence_score:.4f} score falls into the {'excellent' if coherence_score > 0.6 else 'very good' if coherence_score > 0.5 else 'good' if coherence_score > 0.4 else 'moderate'} range.
- The identified topics are {'highly' if coherence_score > 0.5 else 'reasonably'} coherent and 
  semantically meaningful.
- The preprocessing steps (removing stopwords, stemming, lemmatization) and 
  parameter tuning (passes=20, no_below=5, no_above=0.6) contributed to the 
  model's performance.
- The 4 topics extracted from the news dataset represent distinct themes that 
  can be interpreted based on the top terms shown above.

For topic modeling, a higher coherence score indicates that the words within 
each topic tend to co-occur together in the corpus, making the topics more 
interpretable and useful for downstream tasks.
""")


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



Removed 218 null values

Preprocessing documents...
